In [ ]:
import os
import pandas as pd
import json

%matplotlib inline

pd.options.display.max_columns = None
pd.options.display.max_colwidth = 60
pd.options.display.float_format = "{:,.3f}".format

from utils import (
    setup_plot_style,
    fetch_stats,
)

setup_plot_style()
%config InlineBackend.figure_format = 'retina'

In [ ]:
dir_path_base = os.path.expanduser("~/tunable-magmax")

task_seq = "A"
n_splits = 20  # 5 20 50
model = "ViT-B-16"  # __pretrained__laion400m_e31 ViT-L-14
dataset = "CIFAR100"  # ImageNetR CIFAR100
seed_list = [3, 4, 5]
lambda_ = 0.5

dir_name = "merging_target_data_cameraready"
dir_path_shared = f"{dir_path_base}/logs/{model}/sequential_finetuning/class_incremental/{dir_name}/{dataset}-{n_splits}/taskseq_{task_seq}"

mode = "similarity_methods"  # similarity_methods altogether
suffix = f"{mode}_cameraready_debug"  # main_results
dir_name_csv = f"processed_{suffix}"
dir_name_fig = f"figs_{suffix}"
dir_name_stats = f"stats_{suffix}"

if "similarity_methods" in suffix:
    competitor_dict = {
        # "finetune": "Baseline",
        # "merge_rnd_mix": "Random Mix",
        # "sum": "Average",
        # "ties": "TIES-Merging",
        "merge_max_abs": "MAGMAX",
        "merge_max_abs_masked_with_targetdata-cosine_embedded": "Tunable MAGMAX (Cosine)",
        "merge_max_abs_masked_with_targetdata-ot_embedded": "Tunable MAGMAX (OT)",
        "merge_max_abs_masked_with_targetdata-mmd_embedded": "Tunable MAGMAX (MMD)",
        # "select_one_task_vector": "Single task vector",
        "merge_max_abs_masked_with_targetdata-labels": "Tunable MAGMAX (Labels)",
        # "merge_max_abs_masked_with_targetdata-hpo": "Tunable MAGMAX (HPO)",
    }
else:
    competitor_dict = {
        "finetune": "Baseline",
        "merge_rnd_mix": "Random Mix",
        "sum": "Average",
        "ties": "TIES-Merging",
        "merge_max_abs": "MAGMAX",
        # "merge_max_abs_masked_with_targetdata-cosine_embedded": "Tunable MAGMAX (Cosine)",
        "merge_max_abs_masked_with_targetdata-ot_embedded": "Tunable MAGMAX (OT)",
        # "merge_max_abs_masked_with_targetdata-mmd_embedded": "Tunable MAGMAX (MMD)",
        "merge_max_abs_masked_with_targetdata-labels": "Tunable MAGMAX (Labels)",
        # "select_one_task_vector": "Single task vector",
        # "merge_max_abs_masked_with_targetdata-hpo": "Tunable MAGMAX (HPO)",
    }

In [ ]:
with open(f"{dir_path_base}/configs/target_data_config.json", "r") as f:
    target_data_configs = json.load(f)["dataset_configs"]

In [ ]:
results_df = pd.DataFrame(
    index=competitor_dict.keys(), columns=[f"e{i}" for i in range(1, 6)]
)
acc_df = pd.DataFrame(
    index=competitor_dict.keys(), columns=[f"e{i}" for i in range(1, 6)]
)

for i, config in enumerate(target_data_configs):
    if config["num_task_to_be_fetched"] < 0:
        num_task_to_be_fetched, ratio_task_to_be_fetched = "all", ""
    else:
        num_task_to_be_fetched = config["num_task_to_be_fetched"]
        ratio_task_to_be_fetched = f"_with_{'_'.join([str(r) for r in config['ratio_task_to_be_fetched']])}_ratio"

    targetdata_id = [variant["target_id"] for variant in config["variants"]]
    print(
        f"Processing target data config: num_task_to_be_fetched={num_task_to_be_fetched}, ratio_task_to_be_fetched={ratio_task_to_be_fetched}, targetdata_id={targetdata_id}"
    )

    if num_task_to_be_fetched == "all":
        continue
    else:
        mean_dict, std_dict = fetch_stats(
            targetdata_id,
            competitor_dict,
            dir_path_shared,
            seed_list,
            lambda_,
            dir_name,
        )

    for key in competitor_dict.keys():
        results_df.loc[key, f"e{i + 1}"] = f"{mean_dict[key]:.2f} ± {std_dict[key]:.2f}"
        acc_df.loc[key, f"e{i + 1}"] = mean_dict[key]

In [ ]:
results_df["average"] = [round(x, 2) for x in acc_df.mean(axis=1)]

results_df.index = [competitor_dict[key] for key in results_df.index]
save_path = f"{dir_path_shared}/{dir_name_stats}"
os.makedirs(save_path, exist_ok=True)
results_df.to_csv(f"{save_path}/targetdata_results_{suffix}.csv")

results_df